# 16a How few labels? A prior over maps and image selection (THINGS-EEG1 development people)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 08a and 08b (only the 22 development people are opened).

Development analysis, not a prespecified test. The map's gain is smallest at small k. Two ideas are scored against it: a prior over maps (every other person's
map gives a mean and a covariance, and the new person's map is pulled towards that mean along that covariance), at
k = 50, 100, 250, 500; and which images to label (k-centre selection in the feature space and in the space of the
borrowed prediction, against random images) at k = 50, 100, 250. No EEG of the new person is used to choose.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '16a-fewlabels', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"k_prior": [50, 100, 250, 500], "k_select": [50, 100, 250], "primary_ks": [50, 100], "draws": [13, 14, 15], "donor_k": 2000, "donor_draw": 13, "feature_components": 100, "prediction_components": 50, "window_s": [0.1, 0.2], "map_window_ms": 100, "dataset": "EEG1"}')
TAG, DATASET = '16a', 'EEG1'
PEOPLE = ['sub-04', 'sub-07', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-17', 'sub-20', 'sub-21', 'sub-24', 'sub-25', 'sub-26', 'sub-29', 'sub-30', 'sub-34', 'sub-35', 'sub-38', 'sub-39', 'sub-46', 'sub-48', 'sub-49']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import image_selection as sel
from cross_subject_encoding.eval import map_prior as mp
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
assert sha256_file(C.find_artifact("eeg1_paper_kpca1000.npz")) == "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f"
with np.load(C.find_artifact("eeg1_paper_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact("eeg1_k_subsets.json").read_text())
people = sorted(PEOPLE)  # the 22 sources only; the 24 confirmation targets are never opened
y_train, bio_test = {}, {}
for s in people:
    for part in ("training", "test"):
        rel = te.load_part(source_file(Path("/kaggle/input"), s, part, people))
        if part == "training":
            y_train[s], times = rel["data"][:, 0], rel["times"]
        else:
            bio_test[s] = rel["data"]
order = lambda s, d, k: np.asarray(subsets["orders"][s][str(d)])[:k]
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = 17, 100
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
src_sum = src_train.sum(0, dtype=np.float64)
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
win = ha.window_index(times, FIXED["map_window_ms"])
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
feat_space = sel.feature_space(z_train, FIXED["feature_components"])
rows, chosen_rows, spread_rows, curves = [], [], [], {}
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train = (src_sum - src_train[i]) / len(others)
    avg_t3 = t3(src_test[others].mean(0), N_TEST)
    # prior donors: each other person's map, fitted on their own donor_k images against the average of the people who
    # are neither that donor nor the target, so no target data enter the prior
    donor_maps = []
    for j in others:
        idx = order(people[j], FIXED["donor_draw"], FIXED["donor_k"])
        ref = (src_sum[idx] - src_train[i][idx] - src_train[j][idx]) / (len(people) - 2)
        donor_maps.append(h4b.fit_map(t3(ref, len(idx)), np.asarray(y_train[people[j]][idx], np.float64), win,
                                      seed=FIXED["donor_draw"])[0])
    prior = mp.prior_from_maps(donor_maps)
    preds = {"avg|all|all|-": avg_t3, "generic|all|all|-": h4b.apply_map({w: m for w, (m, _) in prior.items()}, avg_t3, win)}
    pred_space = sel.prediction_space(avg_train, FIXED["prediction_components"])  # image-only: sources' predictions
    y_p = y_train[person]
    for d in FIXED["draws"]:
        policies = {"random": order(person, d, max(FIXED["k_prior"])),
                    "kcentre_features": sel.k_center(feat_space, max(FIXED["k_select"]), d),
                    "kcentre_prediction": sel.k_center(pred_space, max(FIXED["k_select"]), d)}
        for k in FIXED["k_prior"]:
            idx = policies["random"][:k]
            a_k, y_k = t3(avg_train[idx], k), np.asarray(y_p[idx], np.float64)
            preds[f"h4b|{k}|{d}|random"] = h4b.apply_map(h4b.fit_map(a_k, y_k, win, seed=d)[0], avg_t3, win)
            for name, gammas in (("mean_prior", (1.0,)), ("cov_prior", mp.GAMMAS)):
                coefs, chosen = mp.fit_map_prior(a_k, y_k, win, prior, gammas=gammas, seed=d)
                preds[f"{name}|{k}|{d}|random"] = h4b.apply_map(coefs, avg_t3, win)
                if name == "cov_prior":
                    chosen_rows += [{"person": person, "k": k, "draw": d, "window": w, "alpha": al, "gamma": gm}
                                    for w, (al, gm) in chosen.items()]
        for k in FIXED["k_select"]:
            for pol, o in policies.items():
                idx = o[:k]
                if pol != "random":
                    a_k, y_k = t3(avg_train[idx], k), np.asarray(y_p[idx], np.float64)
                    preds[f"h4b|{k}|{d}|{pol}"] = h4b.apply_map(h4b.fit_map(a_k, y_k, win, seed=d)[0], avg_t3, win)
                    for space_name, space in (("features", feat_space), ("prediction", pred_space)):
                        spread_rows.append({"person": person, "k": k, "draw": d, "policy": pol, "space": space_name,
                                            "spread_vs_random": sel.spread(space, idx, policies["random"][:k])})
                preds[f"own_ridge|{k}|{d}|{pol}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(
                    z_train[idx], np.asarray(y_p[idx]).reshape(k, -1)).predict(z_test)
    preds = {name: np.asarray(p).reshape(N_TEST, n_ch, n_t) for name, p in preds.items()}
    if not all(np.isfinite(p).all() for p in preds.values()):
        raise ValueError(f"{person}: nonfinite prediction")
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, k, d, policy = name.split("|")
        rows.append({"person": person, "method": method, "k": k, "draw": d, "policy": policy,
                     "r_100_200ms": float(curve[:, window].mean())})
        curves.setdefault(f"{method}|{k}|{policy}", {}).setdefault(person, []).append(curve.mean(0))
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(method, k, policy="random"):
    f = frame[(frame.method == method) & (frame.k == str(k)) & (frame.policy == policy)]
    return f.groupby("person")["r_100_200ms"].mean().reindex(people).to_numpy()
avg = per_person("avg", "all", "-")
def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se),
            "positive": int((x > 0).sum()), "n": int(len(x))}
def verdict(c):
    return "continue" if c["lower95"] > 0 else ("stop" if c["upper95"] < 0.003 else "revise")
pk = FIXED["primary_ks"]
summary = {"dataset": DATASET, "people": len(people), "post_hoc": "development", "map_prior": {}, "image_selection": {}}
summary["k_curve"] = {k: {m: float(per_person(m, k).mean()) for m in ("h4b", "mean_prior", "cov_prior")}
                      for k in FIXED["k_prior"]}
summary["avg"], summary["generic_minus_avg"] = float(avg.mean()), ci(per_person("generic", "all", "-") - avg)
cov_gain = np.mean([per_person("cov_prior", k) - per_person("h4b", k) for k in pk], 0)
summary["map_prior"]["primary_cov_minus_h4b_k50_100"] = ci(cov_gain)
summary["map_prior"]["verdict_this_dataset"] = verdict(summary["map_prior"]["primary_cov_minus_h4b_k50_100"])
for k in FIXED["k_prior"]:
    summary["map_prior"][f"k{k}"] = {"cov_minus_h4b": ci(per_person("cov_prior", k) - per_person("h4b", k)),
                               "mean_minus_h4b": ci(per_person("mean_prior", k) - per_person("h4b", k)),
                               "h4b_minus_avg": ci(per_person("h4b", k) - avg)}
cf = pd.DataFrame(chosen_rows)
summary["map_prior"]["gamma_one_share_by_k"] = {int(k): float((g.gamma == 1.0).mean()) for k, g in cf.groupby("k")}
for pol in ("kcentre_features", "kcentre_prediction"):
    gain = np.mean([per_person("h4b", k, pol) - per_person("h4b", k) for k in pk], 0)
    summary["image_selection"][pol] = {"primary_h4b_minus_random_k50_100": ci(gain), "verdict_this_dataset": verdict(ci(gain))}
    for k in FIXED["k_select"]:
        summary["image_selection"][pol][f"k{k}"] = {"h4b_minus_random": ci(per_person("h4b", k, pol) - per_person("h4b", k)),
                                       "own_ridge_minus_random": ci(per_person("own_ridge", k, pol) - per_person("own_ridge", k))}
sf = pd.DataFrame(spread_rows)
summary["image_selection"]["spread_vs_random_median"] = {f"{p}|{s}": float(g.spread_vs_random.median()) for (p, s), g in sf.groupby(["policy", "space"])}
summary["rule"] = ("development: per idea, continue if the primary gain (k = 50 and 100 averaged) has a 95% lower bound "
                   "above 0 on both datasets; stop if both upper bounds are below +0.003; otherwise revise. "
                   "verdict_this_dataset is this dataset's half of that rule.")
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
np.savez_compressed(C.METRICS_DIR / f"{TAG}_curves.npz", times=times, people=np.array(people),
                    **{name.replace("|", "__"): np.stack([np.mean(by[p], 0) for p in people]) for name, by in curves.items()})
fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
ks = FIXED["k_prior"]
ax[0].axhline(avg.mean(), color="#7f7f7f", ls="--", label="others' average")
for m, c in (("h4b", "#d62728"), ("mean_prior", "#ff7f0e"), ("cov_prior", "#1f77b4")):
    ax[0].plot(range(len(ks)), [summary["k_curve"][k][m] for k in ks], "o-", color=c, label=m)
ax[0].set(xticks=range(len(ks)), xticklabels=[str(k) for k in ks], xlabel="labelled images k", ylabel="r, 100-200 ms",
          title=f"{DATASET}: prior over maps")
ax[0].legend(fontsize=8)
kss = FIXED["k_select"]
for pol, c in (("random", "#d62728"), ("kcentre_features", "#2ca02c"), ("kcentre_prediction", "#9467bd")):
    ax[1].plot(range(len(kss)), [per_person("h4b", k, pol).mean() for k in kss], "o-", color=c, label=f"H4b, {pol}")
    ax[1].plot(range(len(kss)), [per_person("own_ridge", k, pol).mean() for k in kss], "o:", color=c, alpha=0.6)
ax[1].set(xticks=range(len(kss)), xticklabels=[str(k) for k in kss], xlabel="labelled images k", ylabel="r, 100-200 ms",
          title="Which images to label (dotted: own ridge)")
ax[1].legend(fontsize=7)
for j, (name, x) in enumerate((("cov prior - H4b", cov_gain),
                               ("k-centre features - random", np.mean([per_person("h4b", k, "kcentre_features") - per_person("h4b", k) for k in pk], 0)),
                               ("k-centre prediction - random", np.mean([per_person("h4b", k, "kcentre_prediction") - per_person("h4b", k) for k in pk], 0)))):
    ax[2].scatter(np.full(len(x), j) + np.random.default_rng(j).uniform(-0.15, 0.15, len(x)), x, s=14, alpha=0.6)
    ax[2].errorbar(j, x.mean(), yerr=x.std(ddof=1) / np.sqrt(len(x)) * stats.t.ppf(0.975, len(x) - 1), fmt="o", color="black")
ax[2].axhline(0, color="black", lw=0.5)
ax[2].set(xticks=range(3), xticklabels=["cov prior\n- H4b", "k-centre feat.\n- random", "k-centre pred.\n- random"],
          ylabel="gain per person, k = 50-100", title="Primary contrasts (95% CI)")
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({k: summary[k] for k in ("map_prior", "image_selection")}, indent=1)[:4000])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-16a-outputs", title="XSUB 16a few labels outputs",
                  files=[C.METRICS_DIR / f"{TAG}_{x}" for x in ("summary.json", "scores.csv", "curves.npz", "figure.png")],
                  include_dirs=[], message="prior over maps and image selection")